# One request through the whole stack: from the agent to the lakehouse and back


This notebook follows one customer email through the whole stack, from the Foundry agent down to the lakehouse tables and back. Every chapter does the same four things:

1. A diagram at the top shows the components that the request has reached so far. Green boxes are done, the orange box is the component of this chapter.
2. The component is explained in a few sentences, with links to the Microsoft documentation.
3. The relevant configuration files and code are printed from the repository, so what you read is what is deployed.
4. Live evidence shows what the component did for this one request.

The story line is a single email: the customer of project P-200 writes that its forecast shows total costs of EUR 5.75 million and asks us to explain the difference. To answer, the agent needs the project budget, the planned costs, the clauses of the contract documents and the business rules. That touches every layer.

Prerequisites: `az login` with access to the resource group and the Fabric workspace, `jq`, `curl`. Chapters 6 and 7 read the Fabric workspace monitoring logs. If workspace monitoring is not switched on, those cells report that and the rest still works. See [Enable workspace monitoring](https://learn.microsoft.com/fabric/get-started/enable-workspace-monitoring).

Mermaid diagrams render on GitHub and in VS Code with a Mermaid extension. Without one you see the diagram source.

The whole notebook takes 10 to 20 minutes, mostly because Fabric limits the rate of ontology calls and the logs arrive a few minutes after the call.


## 0. Setup

The next cell defines helper functions and looks up the IDs of everything that the chapters use. All cells share one Bash process, so the variables carry over.


In [1]:
set +H   # no history expansion, the KQL below contains exclamation marks
ROOT=$(cd .. && pwd)
RG=rg-projplan-demo
AGENT=project-planning-agent

tok() { az account get-access-token --resource "$1" --query accessToken -o tsv; }
output() { az deployment group show -g "$RG" -n main --query "properties.outputs.$1.value" -o tsv; }
fabric() { curl -s -H "Authorization: Bearer $(tok https://api.fabric.microsoft.com)" "https://api.fabric.microsoft.com/v1$1"; }

# Helpers that print files and code from the repository
show()      { echo "── ${1#$ROOT/} ──"; cat "$1"; echo; }
lines()     { echo "── ${1#$ROOT/} (lines $2 to $3) ──"; sed -n "$2,$3p" "$1"; echo; }
py_func()   { echo "── ${1#$ROOT/}: $2 ──"; awk -v n="$2" '$0 ~ "^def " n "\\(" {p=1; print; next} p && /^(def |# ---|if __name__)/ {exit} p' "$1"; echo; }
bicep_res() { echo "── infra/main.bicep: resource $2 ──"; awk -v n="$2" '$0 ~ "^resource " n " " {p=1} p {print} p && /^}/ {exit}' "$1"; echo; }

PROJECT_ENDPOINT=$(output foundryProjectEndpoint)
ACCOUNT=$(output foundryAccountName)
CAPACITY=$(output fabricCapacityName)
SUB=$(az account show --query id -o tsv)
WS=$(fabric /workspaces | jq -r '.value[] | select(.displayName == "ProjectPlanning") | .id')
ONT=$(fabric /workspaces/$WS/ontologies | jq -r '.value[] | select(.displayName == "ProjectPlanning") | .id')
SM=$(fabric /workspaces/$WS/semanticModels | jq -r '.value[] | select(.displayName == "ProjectPlanningModel") | .id')
LH=$(fabric /workspaces/$WS/lakehouses | jq -r '.value[] | select(.displayName == "projectdata") | .id')
SQL_SERVER=$(fabric /workspaces/$WS/lakehouses/$LH | jq -r '.properties.sqlEndpointProperties.connectionString')
read KQL_DB KQL_URI < <(fabric /workspaces/$WS/kqlDatabases | jq -r '.value[] | select(.displayName == "Monitoring KQL database") | "\(.id) \(.properties.queryServiceUri)"')

kql() {
  curl -s -X POST "$KQL_URI/v1/rest/query" -H "Authorization: Bearer $(tok https://kusto.kusto.windows.net)" \
    -H "Content-Type: application/json" -d "$(jq -n --arg db "$KQL_DB" --arg q "$1" '{db: $db, csl: $q}')"
}

# getDefinition returns the definition at once or as a long running operation
get_definition() {
  local out status loc
  out=$(curl -s -i -X POST -H "Content-Length: 0" -H "Authorization: Bearer $(tok https://api.fabric.microsoft.com)" "https://api.fabric.microsoft.com/v1/workspaces/$WS/$1/getDefinition")
  status=$(echo "$out" | head -1 | awk '{print $2}')
  if [ "$status" = 202 ]; then
    loc=$(echo "$out" | grep -i '^location:' | tr -d '\r' | awk '{print $2}')
    for i in $(seq 40); do
      [ "$(curl -s -H "Authorization: Bearer $(tok https://api.fabric.microsoft.com)" "$loc" | jq -r .status)" = Succeeded ] && break
      sleep 3
    done
    curl -s -H "Authorization: Bearer $(tok https://api.fabric.microsoft.com)" "$loc/result"
  else
    echo "$out" | sed '1,/^\r*$/d'
  fi
}
part() { echo "$1" | jq -r --arg p "$2" '.definition.parts[] | select(.path == $p) | .payload' | base64 -d; }

SQLCMD=$ROOT/.tools/sqlcmd
if [ ! -x "$SQLCMD" ]; then
  mkdir -p "$ROOT/.tools"
  curl -sL https://github.com/microsoft/go-sqlcmd/releases/download/v1.10.0/sqlcmd-linux-amd64.tar.bz2 | tar xj -C "$ROOT/.tools" sqlcmd
fi

echo "Foundry project:  $PROJECT_ENDPOINT"
echo "Agent:            $AGENT"
echo "Fabric capacity:  $CAPACITY, $(az resource show -g "$RG" -n "$CAPACITY" --resource-type Microsoft.Fabric/capacities --query "[sku.name, properties.state]" -o tsv | paste -sd" ")"
echo "Workspace:        $WS"
echo "Ontology:         $ONT"
echo "Semantic model:   $SM"
echo "Lakehouse:        $LH"
echo "Monitoring DB:    ${KQL_DB:-not found, workspace monitoring is off}"


Foundry project:  https://ai-projplan-axowf3.services.ai.azure.com/api/projects/project-planning


Agent:            project-planning-agent


Fabric capacity:  fcprojplanaxowf3, F4 Active


Workspace:        66d9c2b2-d35e-4555-9e74-926164ce584c


Ontology:         8750de7f-d46f-4dd4-84f4-a7e31f68ff24


Semantic model:   22fb9927-ac28-4092-80c0-0f3d7a56a156


Lakehouse:        22d465ee-676f-4ef8-8655-1a7e506d1916


Monitoring DB:    d0b6b49f-c498-4d35-bb1a-f9fdb1c5798b


## 1. The request

```mermaid
flowchart LR
    E["Customer email<br/>E-5 Cost warning"]:::active
    classDef done fill:#e6f4ea,stroke:#34a853,color:#000
    classDef active fill:#fff4ce,stroke:#f29900,stroke-width:3px,color:#000
```

A customer writes to the project manager. Someone has to decide whether this email is critical for the project. The email is a row in [data/emails.csv](../data/emails.csv). The agent does not read that file. The email text is part of the question that we send to it, together with a request for a structured answer.

The cell below defines the question. Everything after it is the journey of this one text.


In [2]:
head -1 "$ROOT/data/emails.csv"; grep "^E-5," "$ROOT/data/emails.csv"; echo

FROM="client@isar.example"
SUBJECT="Cost warning"
BODY="Our current forecast shows total costs of EUR 5.75 million. Please explain the difference."

QUESTION="We received the email below. Answer in this structure:
1. Project and project manager.
2. Verdict: critical, watch or not critical, according to the business rules in the ontology.
3. Why: name the rule that applies and quote the clause of the specification, offer or contract that it touches.
4. Next steps: a numbered list, each step with the responsible person and the deadline.

From: $FROM
Subject: $SUBJECT
Body: $BODY"

echo "$QUESTION"


MailId,ProjectId,Sender,Subject,Body,ReceivedAt


E-5,P-200,controlling@isar.example,Cost warning,Our current forecast shows total costs of EUR 5.75 million. Please explain the difference.,2026-11-05


We received the email below. Answer in this structure:


1. Project and project manager.


2. Verdict: critical, watch or not critical, according to the business rules in the ontology.


3. Why: name the rule that applies and quote the clause of the specification, offer or contract that it touches.


4. Next steps: a numbered list, each step with the responsible person and the deadline.


From: client@isar.example


Subject: Cost warning


Body: Our current forecast shows total costs of EUR 5.75 million. Please explain the difference.


## 2. The Foundry agent

```mermaid
flowchart LR
    E["Customer email"]:::done -->|"question"| A
    subgraph Azure["Azure, Microsoft Foundry"]
        A["Prompt agent<br/>gpt-4.1"]:::active
    end
    classDef done fill:#e6f4ea,stroke:#34a853,color:#000
    classDef active fill:#fff4ce,stroke:#f29900,stroke-width:3px,color:#000
```

The first component is a prompt agent in Microsoft Foundry. It is a model deployment (`gpt-4.1`), a short system prompt and one tool. It holds no business logic. Everything it knows about projects, clauses and ratings comes from the tool at run time.

Documentation: [Foundry Agent Service](https://learn.microsoft.com/azure/foundry/agents/overview), [runtime components](https://learn.microsoft.com/azure/foundry/agents/concepts/runtime-components).

Configuration, part one: the Azure resources that Bicep creates. The Foundry account holds the model deployment and the project. `disableLocalAuth` means there are no API keys, every call uses a Microsoft Entra token.


In [3]:
bicep_res "$ROOT/infra/main.bicep" account
bicep_res "$ROOT/infra/main.bicep" model
bicep_res "$ROOT/infra/main.bicep" project


── infra/main.bicep: resource account ──


resource account 'Microsoft.CognitiveServices/accounts@2026-03-01' = {


  name: accountName


  location: location


  kind: 'AIServices'


  sku: { name: 'S0' }


... (24 outputs left out)

  properties: {


    displayName: 'Project Planning'


  }


}


Configuration, part two: the agent definition. `setup_agent` in [setup_agent.py](../scripts/setup_agent.py), called by [setup.py](../scripts/setup.py), creates a new version of the agent with the model, the prompt from [instructions.txt](../agents/fabric-iq/instructions.txt) and one `FabricIQPreviewTool`. `require_approval="never"` lets the agent call the tool without a human in the loop, which is fine for a read-only tool.

The prompt is short on purpose. The second sentence exists because the model writes the tool request itself and sometimes dropped the sender address, so Fabric could not find the project.


In [4]:
show "$ROOT/agents/fabric-iq/instructions.txt"
py_func "$ROOT/scripts/setup_agent.py" setup_agent


── scripts/agent_instructions.txt ──


You are a project planning assistant for construction projects.


Answer every question with the Fabric IQ tool (the project ontology). Cite the documents or clauses you rely on.


When you call the tool, write a complete question and include the details of the user's message word for word, such as sender addresses, subjects, names and dates. Do not summarize or leave out details, because the tool needs them to find the project.


Do not invent data. If the ontology does not contain the information, say so.


── scripts/setup.py: setup_agent ──


... (16 outputs left out)

        with client.get_openai_client(agent_name=agent.name) as openai:


            question = "Which emails are critical for the project? Rate each one and name the project manager."


            print(f"\nTest question: {question}\n")


            print(openai.responses.create(input=question).output_text)


Now we send the question to the agent through the Responses API. The cell takes the time of the first attempt and of the last one. Fabric limits the request rate of the ontology endpoint, so a call is sometimes refused with HTTP 429 `RequestBlocked`. In that case the cell waits 70 seconds and tries again. A call counts as usable when it is completed and contains a tool call.


In [5]:
ask_once() {
  curl -s --max-time 900 "$PROJECT_ENDPOINT/openai/v1/responses" \
    -H "Authorization: Bearer $(az account get-access-token --scope https://ai.azure.com/.default --query accessToken -o tsv)" \
    -H "Content-Type: application/json" \
    -d "$(jq -n --arg agent "$AGENT" --arg q "$QUESTION" '{agent_reference: {type: "agent_reference", name: $agent}, input: $q}')"
}

T_START=$(date -u +%Y-%m-%dT%H:%M:%SZ)
for ATTEMPT in 1 2 3 4 5 6; do
  T_FINAL=$(date -u +%Y-%m-%dT%H:%M:%SZ)
  RESPONSE=$(ask_once)
  if echo "$RESPONSE" | grep -q "RequestBlocked" || ! echo "$RESPONSE" | jq -e '.status == "completed" and ([.output[]? | select(.type == "mcp_call")] | length > 0)' >/dev/null 2>&1; then
    echo "Attempt $ATTEMPT: no usable answer (Fabric rate limit), waiting 70 s"; sleep 70
  else
    break
  fi
done
T_END=$(date -u +%Y-%m-%dT%H:%M:%SZ)
echo "Attempts: $ATTEMPT"
echo "First attempt started: $T_START (UTC)"
echo "Last attempt ran:      $T_FINAL to $T_END (UTC)"
echo "Response ID:           $(echo "$RESPONSE" | jq -r '.id // "none"')"


Attempts: 1


First attempt started: 2026-11-08T19:19:33Z (UTC)


Last attempt ran:      2026-11-08T19:19:34Z to 2026-11-08T19:21:02Z (UTC)


Response ID:           resp_0cf47baa85e265ec006af0cb48ed9c8193a14689b447f9d9bf


The response echoes the configuration that Foundry applied: agent version, model, prompt and tool. The list of output items shows what happened inside the call: the tool list, the tool call and the final message.


In [6]:
echo "$RESPONSE" | jq -r '"Agent:  \(.agent_reference.name), version \(.agent_reference.version)\nModel:  \(.model)\nTool:   \(.tools[0].type), connection \(.tools[0].project_connection_id | split("/") | last)\n\nPrompt:\n\(.instructions)"'
echo
echo "Output items of the response:"
echo "$RESPONSE" | jq -r '.output[].type' | uniq -c


Agent:  project-planning-agent, version 5


Model:  gpt-4.1


Tool:   fabric_iq_preview, connection fabric-iq-ontology


Prompt:


You are a project planning assistant for construction projects.


Answer every question with the Fabric IQ tool (the project ontology). Cite the documents or clauses you rely on.


... (2 outputs left out)

Output items of the response:


      1 mcp_list_tools


      1 mcp_call


      1 message


## 3. The tool call and the connection

```mermaid
flowchart LR
    E["Customer email"]:::done --> A
    subgraph Azure["Azure, Microsoft Foundry"]
        A["Prompt agent<br/>gpt-4.1"]:::done
        C["Project connection<br/>fabric-iq-ontology"]:::active
    end
    subgraph Fabric["Microsoft Fabric, workspace ProjectPlanning"]
        M["Ontology MCP endpoint"]:::active
    end
    A -->|"tool call fabric_iq_preview"| C
    C -->|"user token"| M
    classDef done fill:#e6f4ea,stroke:#34a853,color:#000
    classDef active fill:#fff4ce,stroke:#f29900,stroke-width:3px,color:#000
```

The model decides to call its one tool. In Foundry the tool is a Fabric IQ tool that points to a project connection. The connection is the bridge between the two platforms: it names the MCP endpoint of the ontology and says how to authenticate.

Two settings matter. `authType: UserEntraToken` makes Foundry forward the Microsoft Entra token of the calling user, with the Power BI API as audience. Fabric therefore sees the real user, and the user needs access to the Fabric workspace. The ontology endpoint does not accept a service principal. The `target` is the address of the endpoint, built from the workspace and ontology IDs.

Documentation: [Fabric IQ tool](https://learn.microsoft.com/azure/foundry/agents/how-to/tools/fabric-iq), [MCP tools in Foundry](https://learn.microsoft.com/azure/foundry/agents/how-to/tools/model-context-protocol), [ontology MCP server](https://learn.microsoft.com/fabric/iq/ontology/how-to-use-ontology-mcp-server).


In [7]:
py_func "$ROOT/scripts/setup_agent.py" create_connection
echo "The connection as Foundry stores it:"
az rest --method get \
  --url "https://management.azure.com/subscriptions/$SUB/resourceGroups/$RG/providers/Microsoft.CognitiveServices/accounts/$ACCOUNT/projects/project-planning/connections/fabric-iq-ontology?api-version=2025-10-01-preview" \
  | jq '.properties | {category, authType, target, audience, isSharedToAll}'


── scripts/setup.py: create_connection ──


def create_connection(name, target):


    sub, rg, account, project = env("AZURE_SUBSCRIPTION_ID"), env("AZURE_RESOURCE_GROUP"), env("FOUNDRY_ACCOUNT_NAME"), env("FOUNDRY_PROJECT_NAME")


    url = (f"https://management.azure.com/subscriptions/{sub}/resourceGroups/{rg}/providers/Microsoft.CognitiveServices/"


           f"accounts/{account}/projects/{project}/connections/{name}?api-version=2025-10-01-preview")


    body = {"properties": {


... (14 outputs left out)

  "target": "https://api.fabric.microsoft.com/v1/mcp/dataPlane/workspaces/66d9c2b2-d35e-4555-9e74-926164ce584c/items/8750de7f-d46f-4dd4-84f4-a7e31f68ff24/ontologyEndpoint",


  "audience": "https://analysis.windows.net/powerbi/api",


  "isSharedToAll": false


}


The tool is an MCP server with three tools. The response lists them in an `mcp_list_tools` item. Then comes the `mcp_call`: the request that the model wrote itself, and the answer that Fabric returned. Look at the request. The prompt told the model to keep the details word for word, so the sender address and the amount should be in it.


In [8]:
echo "Tools that the endpoint offers:"
echo "$RESPONSE" | jq -r '.output[] | select(.type == "mcp_list_tools") | .tools[]? | "  \(.name)"'
echo
echo "$RESPONSE" | jq -r '.output[] | select(.type == "mcp_call") | "Tool: \(.name)\n\nRequest written by the model:\n\(.arguments | fromjson | .request)\n\nAnswer from Fabric:\n\(.output | fromjson | .response)"'


Tools that the endpoint offers:


  ask_ontology


  list_ontology_rules


  list_ontology_entities


Tool: ask_ontology


Request written by the model:


... (28 outputs left out)

2. **Project Manager + Finance/Controlling** — confirm the cost drivers behind the **EUR 750,000** variance and prepare the explanation to the client.


3. **Project Manager + Client contact** — request **written approval** if the overrun is expected to remain above **10%**.


4. **Commercial/Contract owner** — check whether any mitigation, scope adjustment, or cost reforecast is needed before replying.


If you want, I can also summarize the exact email record that matched this assessment.


## 4. The ontology

```mermaid
flowchart LR
    E["Customer email"]:::done --> A
    subgraph Azure["Azure, Microsoft Foundry"]
        A["Prompt agent<br/>gpt-4.1"]:::done
        C["Project connection"]:::done
    end
    subgraph Fabric["Microsoft Fabric, workspace ProjectPlanning"]
        M["Ontology MCP endpoint"]:::done
        O["Ontology<br/>entity types, relationships, rules"]:::active
    end
    A --> C
    C --> M
    M -->|"ask_ontology"| O
    classDef done fill:#e6f4ea,stroke:#34a853,color:#000
    classDef active fill:#fff4ce,stroke:#f29900,stroke-width:3px,color:#000
```

The endpoint reads the ontology. An ontology is the shared vocabulary of the business: entity types (`Project`, `Email`, `ProjectDocument`), the relationships between them and the rules that apply. It stores no data itself. It describes what the data means.

The ontology is a folder of TMDL files in [ontology/](../ontology). `setup.py` uploads the folder with the Fabric REST API. Three kinds of files matter for this email.

Entity types name the properties, give them descriptions and synonyms, and say which table column backs each property. This is how the endpoint knows that "sender" is `Email.Sender`.

Documentation: [Ontology overview](https://learn.microsoft.com/fabric/iq/ontology/overview), [entity types](https://learn.microsoft.com/fabric/iq/ontology/how-to-create-entity-types), [relationship types](https://learn.microsoft.com/fabric/iq/ontology/how-to-create-relationship-types), [rules](https://learn.microsoft.com/fabric/iq/ontology/how-to-use-rules).


In [9]:
show "$ROOT/ontology/entities/Email.tmdl"
echo "── ontology/entityRelationships.tmdl: the relationships that this email uses ──"
awk '/^\/\/\/ A project is led/,/^$/; /^\/\/\/ An email refers/,/^$/; /^\/\/\/ A document belongs/,/^$/' "$ROOT/ontology/entityRelationships.tmdl"


── ontology/entities/Email.tmdl ──


/// An incoming email about a project.


entity Email


	lineageTag: 2dbdea62-398f-57b3-9d01-12ee97185007


	backingTable: emails


	keyProperty: MailId


... (50 outputs left out)

entityRelationship 'Email About Project'


	lineageTag: 4fd8f93f-8d28-59b6-9b9b-9474db4916bb


	fromEntity: Email


	toEntity: Project


Relationship types connect the entity types: an email is about a project, a document belongs to a project, a project has a manager. They let the endpoint follow the email to the project, its documents and its manager.

Rules are natural-language business requirements linked to relevant concepts. `EmailCriticality` now combines rating criteria with the required escalation to the responsible project manager within 1 working day. The former standalone `EscalationPolicy` has been removed. The current agent has no notification tool, so an escalation recommendation is not proof of delivery. Saved outputs in other cells predate this consolidation.


In [ ]:
show "$ROOT/ontology/rules/EmailCriticality.tmdl"


/// Identify incoming emails that require the responsible project manager's attention.


/// Keep the assessment and its required escalation together as one business policy.


/// The policy defines the required action; sending a notification requires an agent tool.


rule EmailCriticality


	lineageTag: 715b2ef6-b7f8-5506-a153-11dcf03bf6a2


	statement: Every incoming email must be rated critical, watch or not critical by comparing it with its project and the documents of that project. Rate it critical if it changes or threatens the completion date, the budget or the scope of the project, or contradicts an assumption or clause of the specification, offer or contract, even when the remaining schedule buffer would absorb it. Rate it watch if nothing changes today but it could affect the schedule, the cost or the scope later, for example a vague wish or an absence that falls inside the term of the project (between its start date and its end date). Rate it not critical if it is routine information without effect on schedule, cost, scope or contract, for example a status meeting, a request for an existing document, or an absence that lies completely outside the term of the project. Place a date without a year in the next occurrence after the date of the email. For an absence or office-closure notice, resolve the full continuous

... (25 outputs left out)

	ruleReferencedRelationship 'Project Managed By'


	annotation Category = Triage


	annotation Owner = Project management


	annotation Source = Specification, offer and contract


Now we ask Fabric what it holds. These are the same MCP tools that the agent could call: `list_ontology_rules` and `list_ontology_entities`. The statements below are the rule text that Fabric returns, not the text in the files, so this proves that the files arrived.


In [11]:
mcp() {
  curl -s "https://api.fabric.microsoft.com/v1/mcp/dataPlane/workspaces/$WS/items/$ONT/ontologyEndpoint" \
    -H "Authorization: Bearer $(tok https://api.fabric.microsoft.com)" -H "Content-Type: application/json" -H "Accept: application/json, text/event-stream" \
    -d "$(jq -n --arg tool "$1" --argjson args "$2" '{jsonrpc: "2.0", id: 1, method: "tools/call", params: {name: $tool, arguments: $args}}')"
}
echo "== Rules as Fabric returns them"
mcp list_ontology_rules '{}' | jq -r '.result.structuredContent.values[]? | "\(.name): \(.statement)\n"'
echo "== Entity Email as Fabric returns it"
mcp list_ontology_entities '{"entityName": "Email"}' | jq -r '.result.structuredContent.values[0]? | "description:   \(.description)\nsynonyms:      \(.synonyms | join(", "))\nbacking table: \(.backingTable)\nkey:           \(.keyProperties | join(", "))\nproperties:    \(.properties | map(.name) | join(", "))"'


== Rules as Fabric returns them


EmailCriticality: Every incoming email must be rated critical, watch or not critical by comparing it with its project and the documents of that project. Rate it critical if it changes or threatens the completion date, the budget or the scope of the project, or contradicts an assumption or clause of the specification, offer or contract, even when the remaining schedule buffer would absorb it. Rate it watch if nothing changes today but it could affect the schedule, the cost or the scope later, for example a vague wish or an absence that falls inside the term of the project (between its start date and its end date). Rate it not critical if it is routine information without effect on schedule, cost, scope or contract, for example a status meeting, a request for an existing document, or an absence that lies completely outside the term of the project. Place a date without a year in the next occurrence after the date of the email.


EscalationPolicy: Every critical email must be escalated to the project manager of its project within 1 working day.


ChangeRequestPolicy: A customer request outside the specification is a change request. It must be charged separately and must not be absorbed into the fixed price.


== Entity Email as Fabric returns it


description:   An incoming email about a project.


synonyms:      Mail, Message


backing table: emails


key:           MailId


properties:    Body, MailId, ProjectId, ReceivedAt, Sender, Subject


## 5. The binding to the semantic model

```mermaid
flowchart LR
    E["Customer email"]:::done --> A
    subgraph Azure["Azure, Microsoft Foundry"]
        A["Prompt agent<br/>gpt-4.1"]:::done
        C["Project connection"]:::done
    end
    subgraph Fabric["Microsoft Fabric, workspace ProjectPlanning"]
        M["Ontology MCP endpoint"]:::done
        O["Ontology"]:::done
        S["Semantic model<br/>ProjectPlanningModel<br/>DAX measures"]:::active
    end
    A --> C
    C --> M
    M --> O
    O -->|"directQuery binding"| S
    classDef done fill:#e6f4ea,stroke:#34a853,color:#000
    classDef active fill:#fff4ce,stroke:#f29900,stroke-width:3px,color:#000
```

The ontology has no data. Each entity type is bound to a table, and in this setup the tables belong to a Power BI semantic model. A semantic model holds tables, relationships and DAX measures, which are named calculations such as the planned cost of a project. The endpoint asks the model, so a sum or a difference is calculated by the engine and not by a language model. Without it the ontology calculated inexactly.

The binding is the part of the demo that goes beyond the Microsoft documentation. The documentation describes the portal action "Generate Ontology" on a semantic model, and the TMDL reference shows only the lakehouse form. The files here copy what the portal writes: an expression that opens the model through its Power BI address, partitions with `mode: directQuery`, and measures that point to the model measures with `EXTERNALMEASURE`. The evidence is in [microsoft-guidance.md](../docs/microsoft-guidance.md).

Documentation: [Data binding](https://learn.microsoft.com/fabric/iq/ontology/how-to-bind-data), [generate an ontology from a semantic model](https://learn.microsoft.com/fabric/iq/ontology/how-to-generate-from-semantic-models), [metrics](https://learn.microsoft.com/fabric/iq/ontology/how-to-use-metrics).

On the ontology side: the expression that opens the model, and a table with its measures and partition. The placeholders in double braces are filled by `setup.py` with the IDs of this environment.


In [12]:
show "$ROOT/ontology/expressions.tmdl"
show "$ROOT/ontology/tables/projects.tmdl"


── ontology/expressions.tmdl ──


expression 'DirectQuery to AS - {{WORKSPACE_ID}}/{{SEMANTIC_MODEL_ID}}' =


		let


		    Source = AnalysisServices.Database("powerbi://api.powerbi.com/v1.0/myorg/{{WORKSPACE_NAME}}", "{{SEMANTIC_MODEL_NAME}}"),


		    Cubes = Table.Combine(Source[Data]),


		    Cube = Cubes{[Id="Model", Kind="Cube"]}[Data]


... (45 outputs left out)

		mode: directQuery


		source


			entityName: projects


			expressionSource: 'DirectQuery to AS - {{WORKSPACE_ID}}/{{SEMANTIC_MODEL_ID}}'


On the semantic model side: the measures and the model settings. `Planned Cost EUR` and `Budget EUR` are the two numbers that the email is about, and `Budget Variance EUR` is their difference. The amount columns are hidden and `discourageImplicitMeasures` is set, so the only route to a budget or a cost is the measure. The model is a Direct Lake model, the partitions of the next chapter show where it reads from.

Documentation: [TMDL overview](https://learn.microsoft.com/analysis-services/tmdl/tmdl-overview), [semantic model definition](https://learn.microsoft.com/rest/api/fabric/articles/item-management/definitions/semantic-model-definition), [measures](https://learn.microsoft.com/power-bi/transform-model/desktop-measures).


In [13]:
show "$ROOT/semantic_model/definition/model.tmdl"
show "$ROOT/semantic_model/definition/tables/projects.tmdl"
lines "$ROOT/semantic_model/definition/tables/tasks.tmdl" 1 9
show "$ROOT/semantic_model/definition/relationships.tmdl"


── semantic_model/definition/model.tmdl ──


model Model


	culture: en-US


	defaultPowerBIDataSourceVersion: powerBI_V3


	discourageImplicitMeasures


ref table project_managers


... (77 outputs left out)

relationship rel_dependency_predecessor


	isActive: false


	fromColumn: task_dependencies.PredecessorId


	toColumn: tasks.TaskId


The repository is the source, but what counts is what runs in Fabric. The next cell reads both definitions back with `getDefinition`. The measures are in the live semantic model, and the live ontology partition points to the model with `mode: directQuery`.


In [14]:
SM_DEF=$(get_definition semanticModels/$SM)
ONT_DEF=$(get_definition ontologies/$ONT)
echo "== Live semantic model, table tasks: measures"
part "$SM_DEF" definition/tables/tasks.tmdl | grep -E "^\s*measure"
echo
echo "== Live semantic model, table projects: measures"
part "$SM_DEF" definition/tables/projects.tmdl | grep -E "^\s*measure"
echo
echo "== Live ontology, table projects: partition"
part "$ONT_DEF" tables/projects.tmdl | sed -n '/partition/,$p'


== Live semantic model, table tasks: measures


	measure 'Duration Days' = DATEDIFF(MIN(tasks[StartDate]), MAX(tasks[EndDate]), DAY) + 1


	measure 'Planned Cost EUR' = SUM(tasks[CostEur])


== Live semantic model, table projects: measures


	measure 'Budget EUR' = SUM(projects[BudgetEur])


	measure 'Budget Variance EUR' = [Planned Cost EUR] - [Budget EUR]


== Live ontology, table projects: partition


## 6. The DAX query

```mermaid
flowchart LR
    E["Customer email"]:::done --> A
    subgraph Azure["Azure, Microsoft Foundry"]
        A["Prompt agent<br/>gpt-4.1"]:::done
        C["Project connection"]:::done
    end
    subgraph Fabric["Microsoft Fabric, workspace ProjectPlanning"]
        M["Ontology MCP endpoint"]:::done
        O["Ontology"]:::done
        S["Semantic model<br/>DAX measures"]:::active
        WM["Workspace monitoring<br/>Eventhouse, KQL"]:::active
    end
    A --> C
    C --> M
    M --> O
    O -->|"DAX query"| S
    S -. "operation log" .-> WM
    classDef done fill:#e6f4ea,stroke:#34a853,color:#000
    classDef active fill:#fff4ce,stroke:#f29900,stroke-width:3px,color:#000
```

When the endpoint answers a question with numbers, it writes a DAX query against the semantic model. A language model inside the endpoint writes that query, and it tries again when the query fails. The measures are what makes the numbers exact: whatever shape the query has, `[Planned Cost EUR]` is always the same calculation.

Until now this step was invisible. Fabric workspace monitoring changes that. It writes the operations of the semantic models in a workspace into an Eventhouse, a KQL database in the same workspace, and the table `SemanticModelLogs` holds the DAX text of every query. Monitoring is a workspace setting (Workspace settings, Monitoring, add a monitoring Eventhouse). It is not created by `setup.py`, because there is no documented API for it.

Documentation: [Workspace monitoring](https://learn.microsoft.com/fabric/get-started/workspace-monitoring-overview), [Semantic model operations](https://learn.microsoft.com/fabric/enterprise/powerbi/semantic-model-operations).

The log arrives a few minutes after the call. The cell waits for the first finished query of the last attempt. The query is shown as KQL, and the table lists every DAX query that the semantic model ran since the last attempt started. `Failed` means that the engine rejected the query and the endpoint wrote another one.


In [15]:
KQL_OPS="SemanticModelLogs
| where Timestamp >= datetime($T_FINAL) and OperationName == 'QueryEnd'
| project Timestamp, OperationId, DurationMs, Status, EventText
| order by Timestamp asc"
echo "KQL used:"; echo "$KQL_OPS"; echo

OPS='{"Tables":[{"Rows":[]}]}'
for i in $(seq 30); do
  OPS=$(kql "$KQL_OPS")
  [ "$(echo "$OPS" | jq '.Tables[0].Rows | length' 2>/dev/null || echo 0)" -gt 0 ] && break
  echo "No log rows yet, waiting 30 s"; sleep 30
done
sleep 20; OPS=$(kql "$KQL_OPS")

N=$(echo "$OPS" | jq '.Tables[0].Rows | length' 2>/dev/null || echo 0)
echo "DAX queries since the last attempt started: $N"
echo "$OPS" | jq -r '.Tables[0].Rows[]? | "\(.[0][11:19])  op \(.[1][0:8])  \(.[2]) ms  \(.[3])\n    \(.[4] | gsub("\\s+"; " ") | .[0:420])"'


KQL used:


SemanticModelLogs


| where Timestamp >= datetime(2026-11-08T19:19:34Z) and OperationName == 'QueryEnd'


| project Timestamp, OperationId, DurationMs, Status, EventText


| order by Timestamp asc


No log rows yet, waiting 30 s


... (11 outputs left out)

19:20:34  op 3081fedf  141 ms  Succeeded


    EVALUATE FILTER( SELECTCOLUMNS( 'documents', "DocId", 'documents'[DocId], "ProjectId", 'documents'[ProjectId], "DocType", 'documents'[DocType], "Title", 'documents'[Title], "CompletionDate", 'documents'[CompletionDate], "TermStart", 'documents'[TermStart], "TermEnd", 'documents'[TermEnd], "FixedPriceEur", 'documents'[FixedPriceEur], "BudgetEur", 'documents'[BudgetEur], "OverrunApprovalThresholdPercent", 'documents'[O


19:20:37  op 5dc8508f  110 ms  Succeeded


    EVALUATE FILTER( SELECTCOLUMNS( 'documents', "DocId", 'documents'[DocId], "DocType", 'documents'[DocType], "Title", 'documents'[Title], "Content", 'documents'[Content] ), [DocId] = "P-200_contract" ) [WaitTime: 0 ms]


If the first attempt was refused, there were earlier attempts. The count below is how many DAX queries ran between the first attempt and the last one. It is zero when the first call worked.


In [16]:
kql "SemanticModelLogs | where Timestamp between (datetime($T_START) .. datetime($T_FINAL)) and OperationName == 'QueryEnd' | summarize n = count(), n_failed = countif(Status == 'Failed')" | jq -r '.Tables[0].Rows[0]? | "DAX queries from the earlier attempts: \(.[0]), of them failed: \(.[1])"'


DAX queries from the earlier attempts: 0, of them failed: 0


## 7. The lakehouse SQL endpoint

```mermaid
flowchart LR
    E["Customer email"]:::done --> A
    subgraph Azure["Azure, Microsoft Foundry"]
        A["Prompt agent<br/>gpt-4.1"]:::done
        C["Project connection"]:::done
    end
    subgraph Fabric["Microsoft Fabric, workspace ProjectPlanning"]
        M["Ontology MCP endpoint"]:::done
        O["Ontology"]:::done
        S["Semantic model<br/>Direct Lake"]:::done
        L["Lakehouse projectdata<br/>SQL analytics endpoint"]:::active
        WM["Workspace monitoring"]:::done
    end
    A --> C
    C --> M
    M --> O
    O --> S
    S -->|"T-SQL"| L
    S -.-> WM
    classDef done fill:#e6f4ea,stroke:#34a853,color:#000
    classDef active fill:#fff4ce,stroke:#f29900,stroke-width:3px,color:#000
```

The semantic model does not hold a copy of the data. Its tables are Direct Lake partitions that point to the Delta tables of the lakehouse, and the model reaches them through the SQL analytics endpoint of the lakehouse. When a DAX query needs a sum, the engine sends T-SQL to that endpoint.

In the TMDL files this is the expression `DatabaseQuery`, which opens the endpoint with `Sql.Database`, and the partition of each table with `mode: directLake`, `entityName`, `schemaName` and `expressionSource`.

Documentation: [Direct Lake overview](https://learn.microsoft.com/fabric/fundamentals/direct-lake-overview), [lakehouse SQL analytics endpoint](https://learn.microsoft.com/fabric/data-engineering/lakehouse-sql-analytics-endpoint).


In [17]:
show "$ROOT/semantic_model/definition/expressions.tmdl"
echo "── semantic_model/definition/tables/tasks.tmdl: partition ──"
sed -n '/partition/,$p' "$ROOT/semantic_model/definition/tables/tasks.tmdl"


── semantic_model/definition/expressions.tmdl ──


expression DatabaseQuery =


		let


		    database = Sql.Database("{{SQL_SERVER}}", "{{SQL_DATABASE_ID}}")


		in


		    database


... (4 outputs left out)

		source


			entityName: tasks


			schemaName: dbo


			expressionSource: DatabaseQuery


The first view of those T-SQL statements comes from the same monitoring log: the events `DirectQueryBegin` carry the statement text. The `OperationId` ties each statement to the DAX query above. The engine sets it as `client_correlation_id` in the SQL session, so you can follow one query across both layers. A DAX query without a statement in this list was answered from the cache of the engine.


In [18]:
KQL_SQL="SemanticModelLogs
| where Timestamp >= datetime($T_FINAL) and OperationName == 'DirectQueryBegin'
| where EventText has 'SELECT' and not(EventText has 'EngineEdition')
| project Timestamp, OperationId, EventText
| order by Timestamp asc"
SQLROWS=$(kql "$KQL_SQL")
echo "T-SQL statements that the semantic model sent to the lakehouse:"
echo "$SQLROWS" | jq -r '.Tables[0].Rows[]? | (.[2] | gsub("\\s+"; " ")) as $s | "\(.[0][11:19])  op \(.[1][0:8])  client_correlation_id \(try ($s | capture("client_correlation_id\u0027, \u0027(?<c>[^\u0027]+)").c) catch "none")\n    \($s | sub("^ ?EXEC sys.sp_set_session_context[^;]*; ?"; "") | sub("OPTION \\( LABEL.*$"; "") | .[0:330])"'


T-SQL statements that the semantic model sent to the lakehouse:


19:20:18  op 989a6c8a  client_correlation_id 989A6C8A-23E9-467D-B5F0-7208B1A0F43D


    SELECT TOP (1000001) [t0].[ProjectId] AS [c1],[t0].[Name] AS [c2],[t0].[BudgetEur] AS [c5],[t0].[ManagerId] AS [c7],[t2].[Name] AS [c17],[t2].[Email] AS [c18],[t4].[MailId] AS [c38],[t4].[ProjectId] AS [c39],[t4].[ReceivedAt] AS [c43] FROM ( ([dbo].[emails] AS [t4] LEFT OUTER JOIN [dbo].[projects] AS [t0] on ( [t4].[ProjectId] =


19:20:22  op 2eaa4030  client_correlation_id 2EAA4030-E0BA-4914-B78F-C3EA034F9DC7


    SELECT TOP (1000001) 1 AS [c37],[t4].[MailId],[t4].[ProjectId],[t4].[Body],[t4].[ReceivedAt] FROM [dbo].[emails] AS [t4] WHERE ( ( [t4].[Sender] = N'client@isar.example' ) AND ( [t4].[Subject] = N'Cost warning' ) ) 


19:20:24  op 8904518f  client_correlation_id 8904518F-1B69-4450-BC89-4649DA149382


... (7 outputs left out)

19:20:34  op 3081fedf  client_correlation_id 3081FEDF-44F5-46C9-AF20-DD97040F22B3


    SELECT TOP (1000001) [t3].[DocId],[t3].[DocType],[t3].[Title],[t3].[CompletionDate],[t3].[TermStart],[t3].[TermEnd],[t3].[FixedPriceEur],[t3].[BudgetEur],[t3].[PenaltyPercentPerWorkingDay],[t3].[PenaltyCapPercent],[t3].[WithdrawalDelayWeeks],[t3].[ChangeRateEurPerHour],[t3].[OverrunApprovalThresholdPercent] FROM [dbo].[documents


19:20:37  op 5dc8508f  client_correlation_id 5DC8508F-3517-426D-9A3E-EDB2575887A7


    SELECT TOP (1000001) [t3].[DocType],[t3].[Title],[t3].[Content] FROM [dbo].[documents] AS [t3] WHERE ( [t3].[DocId] = N'P-200_contract' ) GROUP BY [t3].[DocType],[t3].[Title],[t3].[Content] 


The second view is the query history of the lakehouse itself, `queryinsights.exec_requests_history` of the SQL analytics endpoint. It shows who ran each statement. The `login_name` is the user of the agent call, which confirms that the Entra token went through all layers. Query Insights can lag behind by a short time.

Documentation: [Query insights](https://learn.microsoft.com/fabric/data-warehouse/query-insights).


In [19]:
T_WINDOW_END=$(date -u -d "$T_END +30 seconds" +%Y-%m-%dT%H:%M:%S)
for i in $(seq 12); do
  SQL_LOG=$("$SQLCMD" -S "$SQL_SERVER" -d projectdata --authentication-method ActiveDirectoryAzCli -W -s " | " -Q "
    SET NOCOUNT ON;
    SELECT CONVERT(varchar(12), start_time, 114) AS start_utc, login_name, row_count AS [rows], total_elapsed_time_ms AS ms,
           LEFT(REPLACE(REPLACE(SUBSTRING(command, CHARINDEX('FROM', command), 400), CHAR(10), ' '), '''''', ''''), 160) AS tables_and_filter
    FROM queryinsights.exec_requests_history
    WHERE start_time BETWEEN CONVERT(datetime2, '${T_FINAL%Z}', 126) AND CONVERT(datetime2, '$T_WINDOW_END', 126)
      AND command LIKE '%FROM%' AND command NOT LIKE '%queryinsights%'
    ORDER BY start_time" | grep -v "^Statement ID")
  [ "$(echo "$SQL_LOG" | wc -l)" -gt 3 ] && break
  echo "No statements visible yet, waiting 20 s"; sleep 20
done
echo "$SQL_LOG"


No statements visible yet, waiting 20 s


No statements visible yet, waiting 20 s


No statements visible yet, waiting 20 s


No statements visible yet, waiting 20 s


No statements visible yet, waiting 20 s


No statements visible yet, waiting 20 s


... (6 outputs left out)

19:20:27.671 ga1@cptazure.org 2 44 FROM  (  SELECT [t0].[ProjectId] AS [c1],SUM([t0].[BudgetEur])  AS [a0] FROM [dbo].[projects] AS [t0] GROUP BY [t0].[ProjectId] )  AS [MainTable] WHERE  (  NOT(


19:20:27.724 ga1@cptazure.org 2 53 FROM  ([dbo].[projects] AS [t0]   LEFT OUTER JOIN   [dbo].[project_managers] AS [t2] on  ( [t0].[ManagerId] = [t2].[ManagerId] ) )  OPTION ( LABEL = N'{"Dataset


19:20:34.299 ga1@cptazure.org 1 118 FROM [dbo].[documents] AS [t3] WHERE  ( [t3].[ProjectId] = N'P-200' )  GROUP BY [t3].[DocId],[t3].[DocType],[t3].[Title],[t3].[CompletionDate],[t3].[TermStart],


19:20:37.399 ga1@cptazure.org 1 103 FROM [dbo].[documents] AS [t3] WHERE  ( [t3].[DocId] = N'P-200_contract' )  GROUP BY [t3].[DocType],[t3].[Title],[t3].[Content] OPTION ( LABEL = N'{"DatasetId":


## 8. The data: tables, PDFs and Content Understanding

```mermaid
flowchart LR
    E["Customer email"]:::done --> A
    subgraph Azure["Azure, Microsoft Foundry"]
        A["Prompt agent<br/>gpt-4.1"]:::done
        C["Project connection"]:::done
        CU["Content Understanding<br/>analyzer projectDocumentTerms"]:::active
    end
    subgraph Fabric["Microsoft Fabric, workspace ProjectPlanning"]
        M["Ontology MCP endpoint"]:::done
        O["Ontology"]:::done
        S["Semantic model"]:::done
        L["Lakehouse projectdata<br/>Delta tables"]:::active
    end
    D["CSV files and 4 PDFs"]:::active
    A --> C
    C --> M
    M --> O
    O --> S
    S --> L
    D --> CU
    CU -->|"extracted terms"| L
    D -->|"CSV load"| L
    classDef done fill:#e6f4ea,stroke:#34a853,color:#000
    classDef active fill:#fff4ce,stroke:#f29900,stroke-width:3px,color:#000
```

At the bottom are the data. Four sources feed the lakehouse tables. Project, manager, task, dependency and email data are small CSV files in [data/](../data). `setup.py` uploads them to OneLake and loads each one into a Delta table with the lakehouse load-table API.

The PDFs, one specification, one offer and two contracts, take another way. Azure Content Understanding reads each PDF with the analyzer in [extraction/document_terms.json](../extraction/document_terms.json) and returns the text plus 12 typed terms such as the fixed price, the assumed permit date, the penalty and the overrun approval threshold. `setup.py` writes one row per PDF into the table `documents`. This is why the email can be compared with "the contract says 5 percent" and not only with a text blob.

Documentation: [Content Understanding overview](https://learn.microsoft.com/azure/ai-services/content-understanding/overview), [analyzer reference](https://learn.microsoft.com/azure/ai-services/content-understanding/concepts/analyzer-reference), [load table API](https://learn.microsoft.com/rest/api/fabric/lakehouse/tables/load-table), [OneLake API access](https://learn.microsoft.com/fabric/onelake/onelake-access-api).

The analyzer definition. It builds on the prebuilt document analyzer and asks for confidence values. Each field says what to extract and ends with "Leave empty if ...", so the model does not invent a value.


In [20]:
jq '{description, baseAnalyzerId, models, config}' "$ROOT/extraction/document_terms.json"
echo "Fields (12):"
jq -r '.fieldSchema.fields | to_entries[] | "  \(.key): \(.value.type)"' "$ROOT/extraction/document_terms.json"
echo
echo "Example, the field that matters for this email:"
jq '.fieldSchema.fields.OverrunApprovalThresholdPercent' "$ROOT/extraction/document_terms.json"


{


  "description": "Extracts the key terms of construction specifications, offers and contracts",


  "baseAnalyzerId": "prebuilt-document",


  "models": {


    "completion": "gpt-4.1",


    "embedding": "text-embedding-3-small"


... (21 outputs left out)

  "type": "number",


  "method": "extract",


  "description": "Percentage by which the budget may be exceeded before the client has to approve in writing. Leave empty if there is no such clause."


}


The code that calls the service and loads the tables. `ensure_analyzer` creates or replaces the analyzer, so the JSON file is the single definition. `documents_csv` turns the PDFs into the table, and `upload_and_load` writes all tables to the lakehouse.


In [21]:
echo "── scripts/content_understanding.py: ensure_analyzer and analyze_file ──"
sed -n '/    def ensure_analyzer/,/        return self._wait(r)\["result"\]/p' "$ROOT/scripts/content_understanding.py"
echo
py_func "$ROOT/scripts/setup.py" documents_csv
py_func "$ROOT/scripts/setup.py" upload_and_load


── scripts/content_understanding.py: ensure_analyzer and analyze_file ──


    def ensure_analyzer(self, analyzer_id, definition_path):


        """Creates the analyzer or replaces it, so that the file in extraction/ is the single definition."""


        r = requests.put(self._url(f"analyzers/{analyzer_id}", allowReplace="true"), headers=self._headers(),


                         data=definition_path.read_bytes())


        if not r.ok:


... (46 outputs left out)

            "mode": "Overwrite",


            "formatOptions": {"format": "Csv", "header": True, "delimiter": ","},


        })


        print(f"  Loaded table {table}")


Now the rows themselves, read from the lakehouse SQL endpoint. The first table shows the budget and the second the planned task costs, the two numbers that the DAX measures compared. The third shows the terms that Content Understanding extracted from the documents of P-200.


In [22]:
sql() { "$SQLCMD" -S "$SQL_SERVER" -d projectdata --authentication-method ActiveDirectoryAzCli -W -s " | " -Q "SET NOCOUNT ON; $1" | grep -v "^Statement ID"; echo; }
echo "== projects.BudgetEur"
sql "SELECT ProjectId, Name, BudgetEur FROM projects ORDER BY ProjectId"
echo "== tasks: planned cost per project (what the measure Planned Cost EUR sums)"
sql "SELECT ProjectId, COUNT(*) AS tasks, SUM(CostEur) AS planned_cost_eur FROM tasks GROUP BY ProjectId ORDER BY ProjectId"
echo "== documents of P-200: terms extracted by Content Understanding"
sql "SELECT DocId, DocType, TermStart, TermEnd, OverrunApprovalThresholdPercent, PenaltyPercentPerWorkingDay, BudgetEur FROM documents WHERE ProjectId = 'P-200'"


== projects.BudgetEur


ProjectId Name BudgetEur


--------- ---- ---------


P-100 Office Building Hamburg 2400000


P-200 Residential Complex Munich 5000000


== tasks: planned cost per project (what the measure Planned Cost EUR sums)


... (4 outputs left out)

== documents of P-200: terms extracted by Content Understanding


DocId DocType TermStart TermEnd OverrunApprovalThresholdPercent PenaltyPercentPerWorkingDay BudgetEur


----- ------- --------- ------- ------------------------------- --------------------------- ---------


P-200_contract Contract 2027-03-01 2028-06-30 10 NULL 5000000


## 9. The way back

```mermaid
flowchart LR
    E["Customer email"]:::done -->|"1 question"| A
    subgraph Azure["Azure"]
        A["Prompt agent<br/>gpt-4.1"]:::active
        C["Project connection"]:::done
        AI["Application Insights"]:::active
        CU["Content Understanding"]:::done
    end
    subgraph Fabric["Microsoft Fabric, workspace ProjectPlanning"]
        M["Ontology MCP endpoint"]:::done
        O["Ontology"]:::done
        S["Semantic model"]:::done
        L["Lakehouse"]:::done
        WM["Workspace monitoring"]:::done
    end
    D["CSV files and PDFs"]:::done
    A --> C --> M --> O --> S --> L
    D --> L
    CU --> L
    L -. "rows" .-> S
    S -. "numbers" .-> O
    O -. "answer text" .-> M
    M -. "tool result" .-> A
    A -->|"2 final answer"| E
    A -. "spans" .-> AI
    S -. "operation log" .-> WM
    classDef done fill:#e6f4ea,stroke:#34a853,color:#000
    classDef active fill:#fff4ce,stroke:#f29900,stroke-width:3px,color:#000
```

The rows go up again. The semantic model returns the numbers to the ontology endpoint, which writes an answer text. The endpoint returns it as the tool result, and the agent model writes the final answer for the user.

The agent side has its own record. Connecting an Application Insights resource to the Foundry project turns on server-side tracing: every call produces spans in the table `AppDependencies`. The Bicep file creates the resource and the connection.

Documentation: [Trace concepts](https://learn.microsoft.com/azure/foundry/observability/concepts/trace-agent-concept), [trace setup](https://learn.microsoft.com/azure/foundry/observability/how-to/trace-agent-setup).


In [23]:
bicep_res "$ROOT/infra/main.bicep" appInsights
bicep_res "$ROOT/infra/main.bicep" tracingConnection


── infra/main.bicep: resource appInsights ──


resource appInsights 'Microsoft.Insights/components@2020-02-02' = {


  name: 'appi-${baseName}-${suffix}'


  location: location


  kind: 'web'


  properties: {


... (16 outputs left out)

      ResourceId: appInsights.id


    }


  }


}


The final answer that the user gets. It names the project and the manager, rates the email according to the rule, quotes a clause and lists next steps.


In [24]:
echo "$RESPONSE" | jq -r '.output[] | select(.type == "message") | .content[].text'
echo
echo "$RESPONSE" | jq -r '"Tokens: \(.usage.input_tokens) in, \(.usage.output_tokens) out"'


1. Project and project manager.


   - Project: P-200 — Residential Complex Munich


   - Project manager: Ben Fischer (ben.fischer@example.com)


2. Verdict: Critical.


3. Why:


   - Rule that applies: EmailCriticality


... (7 outputs left out)

   3. Project Manager + Client contact — request written approval if the overrun is expected to remain above 10%.


   4. Commercial/Contract owner — check whether any mitigation, scope adjustment, or cost reforecast is needed before replying.


(Source: EmailCriticality business rule, contract budget clause; see detailed answer for direct quotations and source document link) .


Tokens: 2135 in, 473 out


The trace of the agent call. The `invoke_agent` span covers the whole call, `execute_tool` is the call to Fabric and `chat` is the model writing the final answer. The cell waits until the traces are ingested, which takes a few minutes.


In [25]:
LA_WORKSPACE=$(output logAnalyticsWorkspaceId)
RESPONSE_ID=$(echo "$RESPONSE" | jq -r .id)

query_logs() {
  az rest --method post --resource https://api.loganalytics.io \
    --url "https://api.loganalytics.io/v1/workspaces/$LA_WORKSPACE/query" \
    --headers "Content-Type=application/json" \
    --body "$(jq -n --arg q "$1" '{query: $q}')"
}
get_spans() {
  SPANS=$(query_logs "AppDependencies | where Properties['gen_ai.response.id'] == '$RESPONSE_ID' | order by TimeGenerated asc | project TimeGenerated, Name, DurationMs, Properties")
}

# The invoke_agent span is written last, so wait for it, then let the other spans arrive
for i in $(seq 30); do
  get_spans
  [ "$(echo "$SPANS" | jq '[.tables[0].rows[]? | select(.[1] | startswith("invoke_agent"))] | length')" -gt 0 ] && break
  echo "Trace not ingested yet, waiting 20 s"; sleep 20
done
sleep 15; get_spans

echo "Response ID: $RESPONSE_ID"
echo "$SPANS" | jq -r '.tables[0].rows[]? | [.[0], .[1], "\(.[2] / 1000 | round) s"] | @tsv' | column -t -s $'\t'


Response ID: resp_0cf47baa85e265ec006af0cb48ed9c8193a14689b447f9d9bf


2026-11-08T19:19:37.3965175Z  invoke_agent project-planning-agent:5             84 s


2026-11-08T19:19:37.3969541Z  execute_tool mcp_fabric-iq-ontology.ask_ontology  81 s


2026-11-08T19:20:58.0131879Z  chat gpt-4.1-2025-04-14                           4 s


Now all layers side by side. The agent side comes from the spans, the Fabric side from the workspace monitoring log. The DAX durations include the SQL time. If the two Fabric numbers are small and the call is slow, the time sits in the language model of the endpoint and in the rate limit, not in the calculation.


In [26]:
span_ms() { echo "$SPANS" | jq "[.tables[0].rows[]? | select(.[1] | startswith(\"$1\")) | .[2]] | add // 0 | floor"; }
TOTAL_MS=$(span_ms invoke_agent); TOOL_MS=$(span_ms execute_tool); CHAT_MS=$(span_ms chat)
DAX_N=$(echo "$OPS" | jq '.Tables[0].Rows | length')
DAX_MS=$(echo "$OPS" | jq '[.Tables[0].Rows[] | .[2] // 0] | add // 0 | floor')
SQL_N=$(echo "$SQLROWS" | jq '.Tables[0].Rows | length')

printf "%-52s %8s\n" "Layer" "Time"
printf "%-52s %6s s\n" "Agent call in total (invoke_agent)" "$((TOTAL_MS / 1000))"
printf "%-52s %6s s\n" "  model writes the final answer (chat)" "$((CHAT_MS / 1000))"
printf "%-52s %6s s\n" "  call to Fabric (execute_tool)" "$((TOOL_MS / 1000))"
printf "%-52s %6s s\n" "    DAX in the semantic model ($DAX_N queries, with their SQL)" "$((DAX_MS / 1000))"
printf "%-52s %8s\n" "    T-SQL statements sent to the lakehouse" "$SQL_N"
echo
echo "The rest of the Fabric call, $(( (TOOL_MS - DAX_MS) / 1000 )) s, is the endpoint itself: the language model that writes the DAX, retries and the answer text."


Layer                                                    Time


Agent call in total (invoke_agent)                       84 s


  model writes the final answer (chat)                    3 s


  call to Fabric (execute_tool)                          80 s


    DAX in the semantic model (7 queries, with their SQL)      3 s


    T-SQL statements sent to the lakehouse                  8


The rest of the Fabric call, 76 s, is the endpoint itself: the language model that writes the DAX, retries and the answer text.


## 10. Summary: one request, nine places

| Chapter | Component | Configuration in the repository | Where to look in the portal | Evidence in this notebook |
|---------|-----------|---------------------------------|-----------------------------|---------------------------|
| 2 | Foundry agent | [infra/main.bicep](../infra/main.bicep), [instructions.txt](../agents/fabric-iq/instructions.txt), `setup_agent` | Azure portal, Foundry account and project, agent | Response echo: version, model, prompt, tool |
| 3 | Connection and tool call | `create_connection` in [setup_agent.py](../scripts/setup_agent.py) | Foundry project, connected resources | `mcp_list_tools`, `mcp_call` with the request the model wrote |
| 4 | Ontology | [ontology/entities](../ontology/entities), [entityRelationships.tmdl](../ontology/entityRelationships.tmdl), [ontology/rules](../ontology/rules) | Fabric, ontology item, entity types, rules | `list_ontology_rules`, `list_ontology_entities` |
| 5 | Binding and semantic model | [ontology/expressions.tmdl](../ontology/expressions.tmdl), [ontology/tables](../ontology/tables), [semantic_model](../semantic_model) | Fabric, semantic model, model view | `getDefinition` of both items |
| 6 | DAX | measures in [semantic_model/definition/tables](../semantic_model/definition/tables) | Fabric, workspace monitoring, KQL database, `SemanticModelLogs` | DAX text, duration, status per query |
| 7 | SQL endpoint | [semantic_model/definition/expressions.tmdl](../semantic_model/definition/expressions.tmdl), Direct Lake partitions | Fabric, lakehouse, SQL analytics endpoint | `DirectQueryBegin` statements, Query Insights with `login_name` |
| 8 | Data | [data](../data), [extraction/document_terms.json](../extraction/document_terms.json), `upload_and_load` | Fabric, lakehouse, tables | Rows read through the SQL endpoint |
| 9 | Return path | appInsights and tracingConnection in Bicep | Azure portal, Application Insights, Logs | Spans and the time per layer |

What this one request showed:

* The agent does not know the business. It forwards the question, and the ontology holds the vocabulary and the rules.
* The numbers are calculated by DAX measures, not by a language model. The DAX text itself is written by a language model inside the endpoint, and it can fail and be rewritten.
* Identity passes through every layer. The lakehouse log shows the user of the agent call.
* Most of the time is spent inside the Fabric endpoint and not in the calculation. The endpoint limits the call rate, and a refused call has to be repeated.
* The undocumented parts are the semantic-model binding by code and the form of the partitions. [microsoft-guidance.md](../docs/microsoft-guidance.md) lists what is documented and what we copied from the portal.

More detail: [how-it-works.md](../docs/how-it-works.md) explains the concepts, [configuration-reference.md](../docs/configuration-reference.md) lists every setting with deep links to the Azure and Fabric portals.
